# TPI — Detección de Parkinson mediante análisis tiempo-frecuencia de voz
**Señales y Sistemas 2026**

Pipeline completo de análisis wavelet.

Cuando llegue PC-GITA: montar Drive y cambiar `DATA_DIR`.

---
 1. Instalación
 2. Imports y config
 3. Preprocesamiento
 4. Wavelet
 5. Features
 6. Dataset
 7. Pruebas

In [ ]:
# torchaudio descartado: redundante con librosa + PyWavelets
# whisper / vosk descartados: transcriben lo que se DICE, no analizan COMO suena
# (objetivo opuesto al TPI: nos interesa COMO suena la voz, no QUE dice)
# scikit-learn agregado: necesario para StandardScaler (normalizar features antes de clasificar)
!pip install librosa PyWavelets scipy matplotlib seaborn pandas scikit-learn -q


In [ ]:
import numpy as np                                    # operaciones vectoriales y matriciales sobre arrays numericos
import pywt                                           # PyWavelets: calcula DWT multinivel y CWT continua
import librosa                                        # carga archivos .wav y recorta silencios automaticamente
import librosa.display                                # funciones auxiliares para graficar espectrogramas y señales
import matplotlib.pyplot as plt                       # generacion de graficos y figuras
import seaborn as sns                                 # graficos estadisticos de alto nivel construidos sobre matplotlib
import pandas as pd                                   # manejo de DataFrames para metadatos del dataset
from pathlib import Path                              # rutas del sistema de archivos independientes del sistema operativo
from sklearn.preprocessing import StandardScaler      # normalizacion Z-score: (x - media) / std por columna

# ── Configuracion visual global para todos los graficos ──────────────────────
plt.rcParams.update({
    'figure.dpi'       : 110,        # resolucion en puntos por pulgada (mayor valor = imagen mas nitida)
    'figure.facecolor' : '#0f1117',  # color del fondo exterior de la figura (gris muy oscuro)
    'axes.facecolor'   : '#1a1d2e',  # color del fondo del area de graficado (azul oscuro)
    'text.color'       : 'white',    # color de todo el texto en los graficos
    'axes.labelcolor'  : 'white',    # color de las etiquetas de los ejes X e Y
    'xtick.color'      : 'white',    # color de los numeros y marcas del eje X
    'ytick.color'      : 'white'     # color de los numeros y marcas del eje Y
})
print('Importaciones OK')


In [ ]:
# ── CONFIGURACION GLOBAL ─────────────────────────────────────────────────────
SR          = 44100   # frecuencia de muestreo de PC-GITA [Hz]: 44100 muestras capturadas por segundo de audio
NIVELES_DWT = 8       # niveles de descomposicion diadica: cada nivel divide el ancho de banda a la mitad

# Bandas frecuenciales resultantes con SR=44100 Hz y 8 niveles:
#
#  j | Aprox (pasa-bajo)   | Detalle (pasa-alto)
# ---+---------------------+--------------------------------------------
#  1 | 0 - 11025   Hz      | 11025  - 22050  Hz
#  2 | 0 -  5512.5 Hz      |  5512.5 - 11025  Hz
#  3 | 0 -  2756.2 Hz      |  2756.2 -  5512.5 Hz
#  4 | 0 -  1378.1 Hz      |  1378.1 -  2756.2 Hz
#  5 | 0 -   689   Hz      |   689   -  1378.1 Hz
#  6 | 0 -   344.5 Hz      |   344.5 -   689   Hz  <- armonicos de F0 de la voz
#  7 | 0 -   172.2 Hz      |   172.2 -   344.5 Hz  <- frecuencia fundamental F0 de la voz (~100-200 Hz)
#  8 | 0 -    86.1 Hz      |    86.1 -   172.2 Hz  <- banda del temblor Parkinson (4-8 Hz cae en cA8)
#
# IMPORTANTE: el temblor parkinsónico (4-8 Hz) queda capturado en la APROXIMACION cA8 (0-86.1 Hz),
# que es el coeficiente [0] de la lista. Es el nivel mas diagnostico para Parkinson.
# Los coeficientes de DETALLE cD8 (86-172 Hz) capturan variaciones rapidas de F0.

# ── Composicion del vector de features (por segmento de 2 s) ─────────────────
#   db4    : (8+1) niveles x 4 estadisticos =  36 features
#   sym4   : (8+1) niveles x 4 estadisticos =  36 features
#   morlet :  64  escalas  x 2 estadisticos = 128 features
#   TOTAL  :                                  200 features
#
# ATENCION: aplicar StandardScaler sobre X ANTES de entrenar cualquier clasificador.
# Sin normalizacion, la energia de la CWT a bajas frecuencias (escalas grandes)
# domina numericamente y sesga modelos basados en distancias o gradientes.

print(f'SR          : {SR} Hz')                            # verifica que la frecuencia de muestreo quedo definida
print(f'Niveles DWT : {NIVELES_DWT}')                      # verifica el numero de niveles de descomposicion
print(f'Features/segmento: {(NIVELES_DWT+1)*4*2 + 64*2}') # calcula la dimension total: 9*4*2 + 64*2 = 200


## Modulo 1 — Carga y preprocesamiento

**Por que librosa:** PyWavelets solo trabaja con arrays numpy. Librosa hace el puente: lee el `.wav` del disco y lo entrega como array. Tambien recorta silencios con una sola llamada.

- Entrada: archivo `.wav`
- Salida: `np.ndarray` shape `(N,)`, float64, normalizado en [-1, 1]

In [ ]:
def cargar_señal(path: Path, sr_objetivo: int = SR) -> tuple:
    """Carga .wav con librosa y normaliza en amplitud. PC-GITA ya esta a 44100 Hz."""
    y, sr = librosa.load(path, sr=sr_objetivo, mono=True)
    # librosa.load: lee el archivo .wav del disco
    # sr=sr_objetivo: remuestrea a 44100 Hz si el archivo tiene otra tasa
    # mono=True: promedia canales si el audio es estereo, devuelve un solo canal

    y = y / (np.max(np.abs(y)) + 1e-9)
    # normaliza la amplitud al rango [-1, 1] dividiendo por el maximo valor absoluto
    # el epsilon 1e-9 evita division por cero en señales completamente silenciosas

    return y.astype(np.float64), sr
    # convierte a float64 para mayor precision numerica en los calculos de la DWT


def recortar_silencios(y: np.ndarray, sr: int, top_db: int = 20) -> np.ndarray:
    """Elimina tramos con energia < top_db dB bajo el maximo.

    CORRECCION aplicada: top_db reducido de 30 a 20 dB.
    Con 30 dB se eliminaban tramos de voz debil que en voces parkinsonianas
    son diagnosticamente relevantes: la hipofonia (voz apagada de baja intensidad)
    es un sintoma tipico de la disartria hipocinetica del Parkinson.
    Con 20 dB solo se eliminan silencios reales, conservando la voz debil.
    """
    intervalos = librosa.effects.split(y, top_db=top_db)
    # librosa.effects.split: detecta todos los intervalos donde la señal supera el umbral de energia
    # top_db=20: conserva cualquier tramo cuya energia este dentro de los 20 dB del pico maximo
    # retorna una lista de pares [inicio, fin] expresados en numero de muestra

    if len(intervalos) == 0:
        return y
    # caso borde: si no hay ningun intervalo activo (señal totalmente silenciosa), devuelve la señal original intacta

    return np.concatenate([y[i:f] for i, f in intervalos])
    # extrae cada tramo activo con slicing y los concatena en una sola señal continua sin silencios


def segmentar(y: np.ndarray, sr: int,
              dur_seg: float = 2.0, solapamiento: float = 0.5) -> list:
    """Ventanas de 2 s con 50% de solapamiento -> 88200 muestras por ventana.

    ADVERTENCIA — riesgo de fuga de datos (data leakage):
    Los segmentos solapados de un mismo archivo comparten muestras entre si.
    Si el split train/test se hace a nivel de SEGMENTO, muestras casi identicas
    aparecen simultaneamente en train y en test, inflando artificialmente las metricas.
    SIEMPRE hacer el split agrupando por identificador de PACIENTE/ARCHIVO,
    y solo segmentar DESPUES de separar los conjuntos.
    """
    largo = int(dur_seg * sr)
    # largo: cantidad de muestras por ventana = 2.0 s * 44100 Hz = 88200 muestras

    paso = int(largo * (1.0 - solapamiento))
    # paso: cuantas muestras se avanza entre ventanas consecutivas
    # con solapamiento=0.5 el paso es 44100 muestras (1 segundo), cada ventana comparte la mitad con la anterior

    return [y[i : i + largo] for i in range(0, len(y) - largo + 1, paso)]
    # genera la lista de segmentos deslizando la ventana de largo a largo
    # la condicion len(y) - largo + 1 garantiza que el ultimo segmento este completo


print('Modulo 1 OK — cargar_señal | recortar_silencios | segmentar')


## Modulo 2 — Descomposicion wavelet

**DWT:** red diadica (s = 2^-j, t = k·2^-j). Solo las aproximaciones se descomponen en cada iteracion.

**CWT Morlet:** transformada continua sobre 64 escalas (50–5000 Hz).

In [ ]:
def dwt_multirresolucion(señal: np.ndarray,
                          wavelet: str = 'db4',
                          niveles: int = NIVELES_DWT) -> list:
    """
    DWT multinivel con PyWavelets (algoritmo de Mallat).
    Descompone la señal en una red diadica de escala s = 2^-j y traslacion t = k * 2^-j.
    Solo las aproximaciones se descomponen en cada iteracion (algoritmo piramidal).

    wavelet: 'db4' (Daubechies 4) o 'sym4' (Symlet 4)
    Salida con J=8: lista de 9 arrays en orden [cA8, cD8, cD7, ..., cD1]
      [0] cA8  aproximacion  0    -  86.1 Hz  <- contiene el temblor 4-8 Hz del Parkinson
      [1] cD8  detalle       86.1 - 172.2 Hz  <- variaciones rapidas de F0
      [2] cD7  detalle      172.2 - 344.5 Hz  <- F0 fundamental de la voz
      [3] cD6  detalle      344.5 -   689 Hz  <- primer armonico de F0
      [4] cD5  detalle        689 -  1378 Hz  <- armonicos medios
      [5] cD4  detalle       1378 -  2756 Hz  <- primer formante F1
      [6] cD3  detalle       2756 -  5512 Hz  <- segundo formante F2
      [7] cD2  detalle       5512 - 11025 Hz  <- fricativas y sibilantes
      [8] cD1  detalle      11025 - 22050 Hz  <- ruido de alta frecuencia
    """
    return pywt.wavedec(señal, wavelet, level=niveles)
    # pywt.wavedec: aplica el banco de filtros diadico de forma recursiva sobre las aproximaciones
    # devuelve la lista [cA_J, cD_J, cD_{J-1}, ..., cD_1] en ese orden


def cwt_morlet(señal: np.ndarray,
               sr: int = SR,
               f_min: float = 50.0,
               f_max: float = 5000.0,
               n_escalas: int = 64) -> tuple:
    """
    CWT con wavelet Morlet compleja (cmor1.5-1.0): modulacion B=1.5, frecuencia central C=1.0.
    Cubre F0, primer formante F1 y segundo formante F2 de la voz (50-5000 Hz).
    Salida: coeffs shape (64, N) complejo | freqs shape (64,) en Hz
    """
    wavelet = 'cmor1.5-1.0'
    # wavelet Morlet compleja: buena resolucion tiempo-frecuencia, ideal para señales oscilatorias como la voz
    # parametros cmor<B>-<C>: B=1.5 es el ancho de banda, C=1.0 es la frecuencia central

    freq_central = pywt.central_frequency(wavelet)
    # obtiene la frecuencia central de la wavelet madre normalizada (adimensional)

    escalas = np.geomspace(
        freq_central * sr / f_max,   # escala minima: corresponde a f_max (5000 Hz)
        freq_central * sr / f_min,   # escala maxima: corresponde a f_min (50 Hz)
        num=n_escalas                 # 64 escalas distribuidas logaritmicamente
    )
    # np.geomspace: genera escalas en progresion geometrica (espaciado log-uniforme en frecuencia)
    # la relacion escala-frecuencia en CWT es: f = freq_central * sr / escala

    coeffs, freqs = pywt.cwt(señal, escalas, wavelet, sampling_period=1.0 / sr)
    # pywt.cwt: calcula la correlacion de la señal con versiones escaladas y trasladadas de la wavelet
    # sampling_period=1/sr convierte las escalas a frecuencias reales en Hz
    # coeffs shape: (n_escalas, len(señal)) = (64, N) — un coeficiente complejo por (escala, tiempo)
    # freqs shape: (n_escalas,) — frecuencia en Hz correspondiente a cada fila de coeffs

    return coeffs, freqs


print('Modulo 2 OK — dwt_multirresolucion | cwt_morlet')


## Modulo 3 — Extraccion de features

Vector de dimension **fija = 200** por segmento.

| Componente | Calculo | Dim |
|---|---|---|
| db4  | 9 niveles x 4 estadisticos | 36 |
| sym4 | 9 niveles x 4 estadisticos | 36 |
| Morlet | 64 escalas x 2 estadisticos | 128 |
| **Total** | | **200** |

In [ ]:
def features_dwt(coeffs: list) -> np.ndarray:
    """4 estadisticos por nivel de descomposicion. Con 8 niveles → 9 x 4 = 36 features."""
    features = []
    for c in coeffs:
        # itera sobre cada nivel: [cA8, cD8, cD7, ..., cD1]
        features += [
            np.mean(np.abs(c)),   # media del valor absoluto: mide la amplitud promedio del nivel (energia media)
            np.std(c),            # desviacion estandar: mide la variabilidad de los coeficientes en el tiempo
            np.sum(c ** 2),       # energia total del nivel: suma de cuadrados (norma L2 al cuadrado)
            np.max(np.abs(c))     # amplitud maxima: detecta picos transitorios dentro del nivel
        ]
    return np.array(features)
    # devuelve un vector 1D de 36 floats (9 niveles * 4 estadisticos)


def features_morlet(coeffs_cwt: np.ndarray) -> np.ndarray:
    """Media temporal y std temporal de la energia por escala. 64 x 2 = 128 features."""
    energia = np.abs(coeffs_cwt) ** 2
    # modulo al cuadrado de los coeficientes complejos: escalar real que representa la densidad de energia
    # shape: (64, N) — 64 filas (escalas/frecuencias), N columnas (instantes de tiempo)

    media_temporal = energia.mean(axis=1)
    # promedio de la energia a lo largo del tiempo para cada escala: shape (64,)
    # captura cuanta energia tiene cada banda de frecuencia en promedio durante el segmento

    std_temporal = energia.std(axis=1)
    # desviacion estandar de la energia a lo largo del tiempo para cada escala: shape (64,)
    # captura CUANTO VARIA la energia en el tiempo: alta std indica modulacion de amplitud
    # (relevante para el temblor: la energia fluctua ritmicamente en voces parkinsonianas)

    return np.concatenate([media_temporal, std_temporal])
    # concatena los dos vectores de 64 elementos: resultado final de 128 features


def construir_vector_features(seg: np.ndarray, sr: int = SR) -> np.ndarray:
    """Aplica las tres familias de features y las concatena. Salida: shape (200,)"""
    f_db4 = features_dwt(dwt_multirresolucion(seg, 'db4'))
    # 36 features de la DWT con wavelet Daubechies 4 (buena localizacion temporal)

    f_sym4 = features_dwt(dwt_multirresolucion(seg, 'sym4'))
    # 36 features de la DWT con wavelet Symlet 4 (mas simetrica que db4, complementaria)

    cwt_c, _ = cwt_morlet(seg, sr=sr)
    # coeficientes de la CWT Morlet: shape (64, N), se descarta el vector de frecuencias (_)

    f_cwt = features_morlet(cwt_c)
    # 128 features de la CWT: media y std temporal de la energia por escala

    return np.concatenate([f_db4, f_sym4, f_cwt])
    # concatenacion final: [36 db4 | 36 sym4 | 128 morlet] = 200 features por segmento


def normalizar_dataset(X_train: np.ndarray, X_test: np.ndarray) -> tuple:
    """
    Aplica normalizacion Z-score columna a columna usando StandardScaler.

    CORRECCION: el vector de 200 features NO esta normalizado al salir de
    construir_vector_features. Las energias de la CWT a bajas frecuencias
    (escalas grandes) pueden ser ordenes de magnitud mayores que las de alta
    frecuencia, lo que sesga clasificadores basados en distancias (KNN, SVM-RBF)
    o en gradientes (redes neuronales).

    El scaler se ajusta SOLO sobre X_train y luego se aplica a X_test,
    evitando que informacion del conjunto de prueba contamine el entrenamiento.

    Salida: (X_train_norm, X_test_norm, scaler)
    """
    scaler = StandardScaler()
    # StandardScaler: para cada columna calcula media y std sobre X_train

    X_train_norm = scaler.fit_transform(X_train)
    # fit: calcula media y std de cada feature sobre el conjunto de entrenamiento
    # transform: aplica (x - media) / std → cada columna queda con media≈0 y std≈1

    X_test_norm = scaler.transform(X_test)
    # aplica la misma transformacion (con los parametros del train) al conjunto de prueba
    # NUNCA llamar fit_transform sobre X_test: usaria estadisticas del test y filtraria informacion

    return X_train_norm, X_test_norm, scaler
    # retorna los datos normalizados y el scaler (necesario para transformar nuevas muestras en produccion)


print('Modulo 3 OK — features_dwt | features_morlet | construir_vector_features | normalizar_dataset')


## Modulo 4 — Construccion del dataset

> **Listo para PC-GITA.** Descomentar el bloque de Drive cuando el dataset este disponible.

```
Mi unidad/pc_gita/
├── PD/   <- pacientes Parkinson  (etiqueta 1)
└── HC/   <- controles sanos      (etiqueta 0)
```

In [ ]:
# ── Configuracion de rutas — adaptar segun el entorno ────────────────────────

# --- CUANDO LLEGUE PC-GITA: descomentar estas lineas ---
# from google.colab import drive
# drive.mount('/content/drive')         # monta Google Drive en /content/drive
# DATA_DIR = Path('/content/drive/MyDrive/pc_gita')  # ruta real del dataset en Drive

DATA_DIR  = Path('data/pc_gita')   # ruta local de desarrollo: TODO reemplazar con ruta real de PC-GITA
LABEL_MAP = {'PD': 1, 'HC': 0}    # mapeo nombre-de-carpeta -> etiqueta numerica (ajustar si las carpetas tienen otro nombre)
# PD: Parkinson's Disease (pacientes con Parkinson) -> etiqueta 1
# HC: Healthy Controls (controles sanos)            -> etiqueta 0

print(f'DATA_DIR: {DATA_DIR}')  # confirma la ruta configurada antes de intentar leer archivos


In [ ]:
def construir_dataset(data_dir: Path = DATA_DIR) -> tuple:
    """
    Itera las carpetas PD/ y HC/ y construye la matriz de features completa.

    Salida:
      X  shape (n_segmentos, 200)  — matriz de features, una fila por segmento
      y  shape (n_segmentos,)      — etiquetas: 0=sano (HC) | 1=Parkinson (PD)
      df DataFrame con columnas [archivo, clase, seg_idx] para trazabilidad

    ADVERTENCIA — fuga de datos (data leakage):
    Esta funcion genera multiples segmentos por archivo. Si se hace el split
    train/test sobre las FILAS de X (segmentos), segmentos del mismo paciente
    apareceran en train y en test simultaneamente, inflando artificialmente
    las metricas de clasificacion (el modelo 'reconoce' al paciente en lugar
    de generalizar). Para evitarlo:
      1. Obtener la lista de archivos unicos y hacer el split a nivel de ARCHIVO.
      2. Llamar esta funcion (o una version modificada) separadamente sobre
         los archivos de train y los de test.
      3. Normalizar con normalizar_dataset() DESPUES del split.
    """
    X_lista, y_lista, meta = [], [], []
    # listas acumuladoras: X_lista para vectores de features, y_lista para etiquetas, meta para trazabilidad

    for clase, etiqueta in LABEL_MAP.items():
        # itera sobre {'PD': 1, 'HC': 0}: primero PD, luego HC
        carpeta = data_dir / clase
        # construye la ruta completa: ej. data/pc_gita/PD

        if not carpeta.exists():
            print(f'  [AVISO] No encontrada: {carpeta}')
            continue
        # si la carpeta no existe (dataset aun no disponible) avisa y salta a la siguiente clase

        archivos = sorted(carpeta.glob('*.wav'))
        # busca todos los archivos .wav dentro de la carpeta y los ordena alfabeticamente
        print(f'  {clase}: {len(archivos)} archivos')

        for archivo in archivos:
            # procesa cada archivo de audio de la clase
            try:
                señal, sr = cargar_señal(archivo)
                # lee el .wav y normaliza la amplitud a [-1, 1]

                señal = recortar_silencios(señal, sr)
                # elimina silencios reales (< 20 dB bajo el maximo) sin tocar la voz debil

                for idx, seg in enumerate(segmentar(señal, sr)):
                    # segmenta la señal en ventanas de 2 s con 50% de solapamiento
                    X_lista.append(construir_vector_features(seg, sr))
                    # extrae el vector de 200 features del segmento y lo agrega a la lista

                    y_lista.append(etiqueta)
                    # agrega la etiqueta de clase (0 o 1) correspondiente al segmento

                    meta.append({'archivo': archivo.name, 'clase': clase, 'seg_idx': idx})
                    # guarda nombre del archivo, clase y numero de segmento para trazabilidad

            except Exception as e:
                print(f'  [ERROR] {archivo.name}: {e}')
                # si un archivo falla (corrupto, formato incorrecto, etc.) avisa y continua con el siguiente

    if not X_lista:
        raise RuntimeError(f'Sin datos en {data_dir}. Verificar que existan las carpetas PD/ y HC/')
    # si no se proceso ningun segmento, lanza error con mensaje descriptivo

    X = np.array(X_lista, dtype=np.float64)
    # convierte la lista de vectores a una matriz numpy shape (n_segmentos, 200)

    y = np.array(y_lista, dtype=np.int32)
    # convierte las etiquetas a array numpy shape (n_segmentos,)

    df = pd.DataFrame(meta)
    # crea un DataFrame con los metadatos: util para hacer el split por paciente correctamente

    print(f'\nDataset: {X.shape[0]} segmentos x {X.shape[1]} features')
    print(f'  PD: {y.sum()}  HC: {(y==0).sum()}')
    # muestra el balance de clases: idealmente debe estar cerca de 50/50
    return X, y, df


# Para correr cuando llegue PC-GITA:
# X, y, df_meta = construir_dataset()
# Luego recordar hacer split por paciente y normalizar con normalizar_dataset()
print('Modulo 4 OK — construir_dataset')


## Prueba de humo — verificacion sin PC-GITA

Señal sintetica: vocal sostenida con temblor de 5 Hz (simula caracteristicas parkinsonianas).

In [ ]:
# ── Señal sintetica: vocal sostenida con tremor de frecuencia modulacion de 5 Hz ──
t = np.linspace(0, 2.0, int(2.0 * SR))
# t: vector de tiempo de 0 a 2 segundos con 88200 puntos (1 punto cada 1/44100 s)

tremor = 0.05 * np.sin(2 * np.pi * 5 * t)
# tremor: oscilacion sinusoidal a 5 Hz con amplitud 0.05 (5% de modulacion)
# simula el temblor de voz parkinsónico (4-8 Hz caracteristico de la disartria hipocinetica)
# NOTA: 5 Hz cae en la banda de la APROXIMACION cA8 (0-86.1 Hz), no en los detalles

voz = np.sin(2 * np.pi * 140 * (1 + tremor) * t) + 0.1 * np.random.randn(len(t))
# voz: vocal sostenida a 140 Hz (F0 tipico de voz masculina) con modulacion de frecuencia por el tremor
# el tremor modula la frecuencia instantanea: F(t) = 140 * (1 + 0.05 * sin(2π*5*t))
# + 0.1 * randn: agrega ruido blanco gaussiano con std=0.1 para simular ruido de fondo real

# ── Aplicar las tres familias de analisis a la señal sintetica ───────────────
c_db4 = dwt_multirresolucion(voz, 'db4')
# descompone la voz con wavelet Daubechies 4 en 9 niveles: [cA8, cD8, ..., cD1]

c_sym4 = dwt_multirresolucion(voz, 'sym4')
# descompone la voz con wavelet Symlet 4 en 9 niveles: alternativa simetrica a db4

c_cwt, f_cwt = cwt_morlet(voz)
# calcula la CWT Morlet sobre 64 escalas entre 50 y 5000 Hz
# c_cwt shape: (64, 88200) — coeficientes complejos para cada (escala, instante)
# f_cwt shape: (64,) — frecuencia en Hz de cada fila de c_cwt

vec = construir_vector_features(voz)
# construye el vector de 200 features completo: [db4(36) | sym4(36) | morlet(128)]

# ── Verificacion de dimensiones ───────────────────────────────────────────────
print(f'DWT db4   — niveles: {len(c_db4)} | muestras por nivel: {[len(c) for c in c_db4]}')
# 9 niveles (1 aprox + 8 detalles); las muestras se reducen a la mitad en cada nivel

print(f'DWT sym4  — niveles: {len(c_sym4)}')
# debe ser identico al de db4 (misma estructura de descomposicion)

print(f'CWT Morlet — rango de frecuencias: {f_cwt[-1]:.1f} – {f_cwt[0]:.1f} Hz')
# f_cwt[-1] es la frecuencia mas baja (50 Hz), f_cwt[0] la mas alta (5000 Hz)

print(f'Vector total: {vec.shape[0]} features  '
      f'(db4={features_dwt(c_db4).shape[0]} | '
      f'sym4={features_dwt(c_sym4).shape[0]} | '
      f'morlet={features_morlet(c_cwt).shape[0]})')
# verifica que las dimensiones parciales suman correctamente: 36 + 36 + 128 = 200

print('\nNOTA: el tremor de 5 Hz queda capturado en cA8 (aprox 0-86 Hz).')
print('      Para verificarlo: np.sum(c_db4[0]**2) deberia ser el nivel de mayor energia.')
print('\nPipeline OK — listo para recibir PC-GITA')


In [ ]:
BANDAS = [
    'Aprox\n0-86 Hz', 'D8\n86-172 Hz', 'D7\n172-344 Hz',
    'D6\n344-689 Hz', 'D5\n689-1378 Hz', 'D4\n1378-2756 Hz',
    'D3\n2756-5512 Hz', 'D2\n5512-11025 Hz', 'D1\n11025-22050 Hz'
]
# etiquetas de los 9 niveles para el eje X: aproximacion mas los 8 niveles de detalle

energias = [np.sum(c**2) for c in c_db4]
# calcula la energia total (suma de cuadrados) de cada nivel de la DWT db4
# la aproximacion (indice 0) concentra la mayor energia por ser pasa-bajo

colores = ['#e07b54' if i <= 3 else '#5ab552' for i in range(len(energias))]
# asigna color naranja a los 4 primeros niveles (aprox + D8 + D7 + D6): bandas clinicamente relevantes
# color verde para los niveles de mayor frecuencia (menos informativos para Parkinson)

fig, ax = plt.subplots(figsize=(11, 4))
# crea una figura de 11x4 pulgadas con un solo eje de graficado

ax.bar(range(len(energias)), energias, color=colores, edgecolor='#2e3250', linewidth=0.8)
# grafico de barras: eje X = indice del nivel, eje Y = energia total del nivel
# edgecolor: borde oscuro entre barras para mejorar la separacion visual

ax.set_xticks(range(len(BANDAS)))
# posiciona un tick en el eje X por cada barra (uno por nivel)

ax.set_xticklabels(BANDAS, fontsize=7.5)
# asigna las etiquetas con nombre y rango de frecuencia a cada tick

ax.set_ylabel('Energia total del nivel')
# etiqueta del eje Y: unidades son amplitud^2 * muestras (energia discreta)

ax.set_title('Energia por nivel DWT — señal sintetica (db4)', color='white')
# titulo del grafico

ax.text(0.5, 0.92,
        'Naranja = bandas relevantes para F0 y temblor Parkinson  |  Verde = alta frecuencia',
        transform=ax.transAxes, ha='center', fontsize=8, color='#e07b54')
# agrega una anotacion centrada en la parte superior del grafico como referencia de colores
# transform=ax.transAxes: coordenadas relativas al eje (0-1), independientes de los datos

plt.tight_layout()
# ajusta automaticamente los margenes para que etiquetas y titulo no se corten

plt.show()
# renderiza y muestra la figura
